## 07 Structured Output

Response from LLMs is unstructured text. One powerful feature of a LangChain agent is its ability to produce structured output. If agents are going to be working with existing computer systems, it is essentially that they are able to produce data in defined formats.


<div align="center">
<img src="images/07_structured_output.png" width="450" heigh="300" alt="Agent with Structured Output"/>
</div>

This notebook shows you how an agent can produce structured output.

In [1]:
from dotenv import load_dotenv
from langchain_community.utilities import SQLDatabase
from dataclasses import dataclass
import langchain

print(f"Using langchain version: {langchain.__version__}")

load_dotenv(override=True)

Using langchain version: 1.2.14


True

## How To
Structured output format usually implied producing output as an instance of a `typing.TypedDict` class. 

Here are the steps you use for the simplest case:

1. Step 1: define the output structure you want. This should include all fields you expect to "parse" from the text output of your LLM.

    ```python
    from typing import TypedDict

    class ContactInfo(TypedDict):
        # here are the fields I want parsed out 
        # from my LLM response
        name: str
        email: str
        phone: str
    ```

2. Step 2: create your agent the usual way, but define the output format with the `response_format` parameter as follows:

    ```python
    from langchain.agents import create_agent

    agent = create_agent(
        model="openai:gpt-5-mini",
        # the following param means output format 
        # will follow the structure of ContactInfo TypeDict
        response_format=ContactInfo,
    )
    ```

3. Step 3: call the agent the usual way

    ```python
    # let's assume I am passing a transcribed version of a
    # recorded conversation, which goes like this
    recorded_conversation = """
        We spoke to John Doe, whoo works at MagicWorks Inc. Let's see,
        his contact number is five, five, five, two, three, one, five,
        four, seven. Hope you got that. And yes, his email id is
        john doe at magicworks dot com. He wanted to order 50 Dragon Heartstring wands"""

    result = agent.invoke(
        {"messages" : [{"role": "user", "content": recorded_conversation}]}
    )
    print(result["structured_response"])
    ```
        

In [8]:
# Step 1 - define the structure of the expected output
from typing import TypedDict


class ContactInfo(TypedDict):
    # here are the fields I want parsed out
    # from my LLM response
    name: str
    email: str
    phone: str
    order_details: str


# Step 2 - create the agent
from langchain.agents import create_agent

agent = create_agent(
    model="openai:gpt-5-mini",
    system_prompt="Extract information from conversation. Extract only requested details and nothing else (no extra text!)",
    # the following param means output format
    # will follow the structure of ContactInfo TypeDict
    response_format=ContactInfo,
)

In [9]:
recorded_conversation = """
    We spoke to John Doe, whox works at MagicWorks Inc. Let's see,
    his contact number is five, five, five, two, three, one, five,
    four, seven. Hope you got that. And yes, his email id is
    john doe at magicworks dot com. He wanted to order 50 Dragon Heartstring wands"""

In [10]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": recorded_conversation,
            }
        ]
    }
)
print(result["structured_response"])

{'name': 'John Doe', 'email': 'john.doe@magicworks.com', 'phone': '555231547', 'order_details': '50 Dragon Heartstring wands'}


Wow! It not only got that perfectly, but neatly captured the output in the desired format.

### Multiple formats supported
Agents support multiple formats for structured output. For example:
* `typing.TypedDict` (as we have used above)
* dataclass (`dataclasses.dataclass` with `@dataclass` annot)
* JSON schema (dict)
* Pydantic `BaseModel`

Let's see the same example, but with a Pydantic Dataclass

In [11]:
from langchain.agents import create_agent
from pydantic import BaseModel


class ContactInfo2(BaseModel):
    # here are the fields I want parsed out
    # from my LLM response
    name: str
    email: str
    phone: str
    order_details: str
    birthday: str


# and our agent now using the above response_format
agent2 = create_agent(
    model="openai:gpt-5-mini",
    system_prompt="Extract information from conversation. Extract only requested details and nothing else (no extra text!)",
    # the following param means output format
    # will follow the structure of ContactInfo TypeDict
    response_format=ContactInfo2,
)

# a conversation, which now also includes a birthday
recorded_conversation2 = """
    We spoke to John Doe, whoo works at MagicWorks Inc. Let's see,
    his contact number is five, five, five, two, three, one, five,
    four, seven. Hope you got that. And yes, his email id is
    john doe at magicworks dot com. He wanted to order 50 Dragon 
    Heartstring wands for his daughter, who was born on the second
    of July 1985.
"""

# let's invoke it on the same conversation
result = agent2.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": recorded_conversation2,
            }
        ]
    }
)
# slightly different way of showing the output (no print)
result["structured_response"]

ContactInfo2(name='John Doe', email='johndoe@magicworks.com', phone='555231547', order_details='50 Dragon Heartstring wands', birthday='1985-07-02')

You can see now that the output follows the Pydantic BaseModel structure we defind above. It has even parsed the birthday in `YYYY-mm-dd` format!

### Conclusion
LLMs produce unstructured text as output. A LangChain agent is its ability to produce structured output conforming to user-defined `response_format`, which can be a `typing.TypedDict` or a `pydantic.BaseModel` or a `dataclasses.dataclass` or a `JSON schema dict`. 

One contraint is that all fields of this format class must be of the `str` type only, which you can then convert to desired type.